# 05 - Baseline Comparison

## Overview

Compare XGBoost and LightGBM centralized baselines on the test set.


In [ ]:
# Imports
import numpy as np
import pandas as pd
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
from src.config import SEED, XGB_PARAMS, LGB_PARAMS

import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score, accuracy_score


# Paths
PREPROCESSED_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed"
MODELS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/models"
LOGS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/logs"
FIGURES_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/figures"


## 1. Load Data

In [ ]:
print("Loading data...")
train_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "train.parquet"))
test_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "test.parquet"))

# Load metadata
with open(os.path.join(PREPROCESSED_DIR, "metadata.json"), 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
num_classes = metadata['num_classes']

# Prepare data
X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df['label'].values.astype(np.int64)
X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df['label'].values.astype(np.int64)

print(f"Test: {X_test.shape}")
print(f"Classes: {num_classes}")

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.utils import resample

print("\n" + "=" * 60)
print("TRAINING XGBOOST & LIGHTGBM")
print("=" * 60)

# Subsample to avoid OOM
sample_size = min(200_000, len(X_train))
X_tr, y_tr = resample(X_train, y_train, n_samples=sample_size, random_state=42)
print(f"Training XGB/LGB on {sample_size:,} samples (was {len(X_train):,})")

# XGBoost
print("\nTraining XGBoost...")
start_time = time.time()

xgb_params = XGB_PARAMS.copy()
xgb_params['n_jobs'] = 8
xgb_params.pop('n_estimators', None)
xgb_model = xgb.XGBClassifier(**xgb_params, n_estimators=100)
xgb_model.fit(X_tr, y_tr, verbose=False)
xgb_time = time.time() - start_time

xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred, average='macro')
print(f"XGBoost completed in {xgb_time:.1f}s")
print(f"XGBoost Test Accuracy: {xgb_acc:.4f}")
print(f"XGBoost Test F1 Macro: {xgb_f1:.4f}")

xgb_model.get_booster().save_model(os.path.join(MODELS_DIR, "centralized_xgboost.ubj"))
print("Model saved: results/models/centralized_xgboost.ubj")

In [ ]:
# LightGBM
print("\nTraining LightGBM...")
start_time = time.time()

lgb_params = LGB_PARAMS.copy()
lgb_params['n_jobs'] = 8
lgb_params.pop('n_estimators', None)
lgb_model = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
lgb_model.fit(X_tr, y_tr)
lgb_time = time.time() - start_time

lgb_pred = lgb_model.predict(X_test)
lgb_acc = accuracy_score(y_test, lgb_pred)
lgb_f1 = f1_score(y_test, lgb_pred, average='macro')
print(f"LightGBM completed in {lgb_time:.1f}s")
print(f"LightGBM Test Accuracy: {lgb_acc:.4f}")
print(f"LightGBM Test F1 Macro: {lgb_f1:.4f}")

lgb_model.booster_.save_model(os.path.join(MODELS_DIR, "centralized_lightgbm.txt"))
print("Model saved: results/models/centralized_lightgbm.txt")

## 3. Comparison Summary


In [ ]:
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM'],
    'Accuracy': [xgb_acc, lgb_acc],
    'F1 Macro': [xgb_f1, lgb_f1],
    'Training Time (s)': [xgb_time, lgb_time]
})

print(comparison.to_string(index=False))


In [ ]:
# Plot comparison
fig, ax = plt.subplots(figsize=(8, 5))

models = ['XGBoost', 'LightGBM']
colors = ['#2ca02c', '#d62728']

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['Accuracy'], width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, comparison['F1 Macro'], width, label='F1 Macro', color='coral')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy & F1 Macro')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '05_baseline_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved: results/figures/05_baseline_comparison.png")


## 4. Save Results


In [ ]:
# Compile results
baseline_results = {
    'models': {
        'XGBoost': {'accuracy': float(xgb_acc), 'f1_macro': float(xgb_f1), 'train_time': float(xgb_time)},
        'LightGBM': {'accuracy': float(lgb_acc), 'f1_macro': float(lgb_f1), 'train_time': float(lgb_time)}
    },
    'best_model': 'XGBoost' if xgb_f1 >= lgb_f1 else 'LightGBM'
}

with open(os.path.join(LOGS_DIR, '05_baseline_results.json'), 'w') as f:
    json.dump(baseline_results, f, indent=2)

print("Baseline results saved: results/logs/05_baseline_results.json")


In [ ]:
print()
print("=" * 60)
print("BASELINE COMPARISON SUMMARY")
print("=" * 60)
print()
print("- XGBoost & LightGBM trained on 200k samples")
print()
print("Results:")
print(f"{'Model':<12} {'Accuracy':>10} {'F1 Macro':>10} {'Time':>8}")
print(f"{'-'*12} {'-'*10} {'-'*10} {'-'*8}")
print(f"{'XGBoost':<12} {xgb_acc:>10.4f} {xgb_f1:>10.4f} {xgb_time:>7.1f}s")
print(f"{'LightGBM':<12} {lgb_acc:>10.4f} {lgb_f1:>10.4f} {lgb_time:>7.1f}s")
